# GRM ladder — rung vs classic GRM

Joins a MAF-ladder rung GRM (built on the 50K subsample) to the classic GRM
(`06_grm_shards.ipynb`, MAF > 1 %, full cohort) **pair by pair**, and plots
classic relatedness on x against binned rung relatedness on y.

Set `RUNG` in the config to `"0p1pct"` or `"5pct"`.

### Nothing is copied to the VM

The classic GRM stays in the bucket and is **streamed** through `gcloud storage
cat`, one shard at a time. Only the small rung triangle is pulled local.

| | samples | lower triangle | handling |
|---|---|---|---|
| classic | 227,561 | ~103 GB | streamed from GCS, never stored |
| rung | 50,000 | ~5 GB | downloaded once, memory-mapped |

Streaming whole shards reads more bytes than range-requesting just the ~22 % of
rows that are needed, but it is faster: 20 sequential requests beat ~50,000
round trips, which would be latency-bound. A pipe cannot seek, so unwanted rows
are read and discarded; reading stops early once a shard's last wanted row has
gone past.

### Why this is not a simple join

The two matrices share neither sample set nor ordering, so row *i* of one is not
row *i* of the other. The approach:

1. Read each GRM's `.grm.id` to recover its own sample ordering.
2. Intersect on `person_id`; order the intersection by **classic** row index, which
   is the order the stream delivers rows in.
3. Memory-map the rung triangle so any pair can be looked up directly.
4. Accumulate binned statistics as rows go past. Peak memory is one classic row.

`plink --make-grm-bin --parallel k N` writes contiguous row blocks, so each
shard's row range is recovered from its size in the bucket rather than parsed
from a log.

## Config

In [ ]:
import os, math, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUNG = "0p1pct"          # "0p1pct" or "5pct"

WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/1kg_eur"

# Both read straight from the bucket — the classic GRM is streamed, not copied.
CLASSIC_GS = f"{R_GS}/03_grm/shards"
RUNG_GS    = f"{R_GS}/03_grm/shards_{RUNG}_50k"

N_SHARDS = 20            # both GRMs were built with --parallel k 20

LOCAL = os.path.expanduser(f"~/scratch_grm_compare_{RUNG}")
os.makedirs(LOCAL, exist_ok=True)
RUNG_TRI = f"{LOCAL}/rung_{RUNG}.tri.bin"     # concatenated rung lower triangle

# Shards are streamed whole, so wall time is set by the ~103 GB transfer rather
# than by how many rows are kept. MAX_ROWS therefore saves little time unless it
# is small enough to skip entire trailing shards — and close-relative pairs thin
# out quadratically, so the high-relatedness bins go sparse fast. Prefer None.
MAX_ROWS = None          # None = every intersection sample

# Binning on the classic axis. Fine enough to resolve the 0.125 / 0.25 / 0.5 spikes.
BIN_LO, BIN_HI, BIN_W = -0.10, 0.60, 0.002
BINS = np.arange(BIN_LO, BIN_HI + BIN_W, BIN_W)

SCATTER_TARGET = 400_000  # raw pairs kept for the density panel
RNG_SEED = 20260927

print(f"rung:    {RUNG_GS}")
print(f"classic: {CLASSIC_GS}  (streamed)")
print(f"bins:    {len(BINS)-1} of width {BIN_W} over [{BIN_LO}, {BIN_HI}]")

## Locate shards and recover row ranges

`--parallel k N` splits by rows, so shard *k* holds rows `[r_{k-1}, r_k)` of the
lower triangle. Row *i* has *i+1* entries, so the cumulative float count through
shard *k* is `T(r_k)` where `T(r) = r(r+1)/2`. Inverting that recovers each
boundary from the object sizes in the bucket — no download, no log parsing.

The final boundary must land exactly on *N* from `.grm.id`. If it does not, the
shard set is incomplete or the split is not what this assumes, and the assertion
stops rather than silently producing a misaligned join.

In [ ]:
def tri_n(r):
    """Number of lower-triangle entries in the first r rows."""
    return r * (r + 1) // 2


def inv_tri_n(t):
    """Largest r with tri_n(r) <= t, exactly (integer sqrt, no float error)."""
    r = (math.isqrt(8 * t + 1) - 1) // 2
    while tri_n(r + 1) <= t:
        r += 1
    while tri_n(r) > t:
        r -= 1
    return r


def gs_listing(prefix):
    """{url: size} for one prefix. Lists rather than stat-ing each object, so a
    partial shard set reports what IS there instead of failing opaquely."""
    r = subprocess.run(["gcloud", "storage", "ls", "-l", prefix],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"cannot list {prefix}\n{r.stderr.strip()}")
    out = {}
    for line in r.stdout.splitlines():
        parts = line.split()
        if len(parts) >= 2 and parts[-1].startswith("gs://"):
            try:
                out[parts[-1]] = int(parts[0])
            except ValueError:
                pass          # the trailing TOTAL: line
    return out


def require_shards(prefix, n_shards, label):
    """
    Resolve shard paths and sizes, reporting precisely what is missing.

    plink 1.9 `--parallel k N` appends `.k` to every output name, so shard 3 of
    `--out foo` is `foo.grm.bin.3`. Older/merged layouts drop the suffix, so both
    are tried.
    """
    listing = gs_listing(prefix.rstrip("/") + "/")

    for suffixed, pat in ((True,  "{p}/grm.shard{k}.grm.bin.{k}"),
                          (False, "{p}/grm.shard{k}.grm.bin")):
        paths = [pat.format(p=prefix.rstrip("/"), k=k)
                 for k in range(1, n_shards + 1)]
        if all(q in listing for q in paths):
            zero = [q for q in paths if listing[q] == 0]
            assert not zero, (
                f"{label}: zero-byte shards "
                f"{[os.path.basename(z) for z in zero]}"
            )
            print(f"{label}: {n_shards} shards, "
                  f"{'plink --parallel .k suffix' if suffixed else 'plain names'}")
            return paths, [listing[q] for q in paths]

    # neither layout is complete — say exactly what is there
    best = [q for q in
            (f"{prefix.rstrip('/')}/grm.shard{k}.grm.bin.{k}"
             for k in range(1, n_shards + 1)) if q in listing]
    print(f"{label}: only {len(best)}/{n_shards} shards found\n")
    if listing:
        bins = {k: v for k, v in listing.items()
                if ".grm.bin" in k and ".grm.N.bin" not in k}
        print(f"GRM shard objects under {prefix}/ :")
        for k in sorted(bins):
            print(f"  {bins[k] / 1024**3:9.3f} GB  {os.path.basename(k)}")
        other = len(listing) - len(bins)
        if other:
            print(f"  (+{other} other objects)")
    else:
        print("that prefix is empty or does not exist")
    raise SystemExit(
        f"{label} GRM is incomplete. If the job is still running, wait; if it "
        f"finished, read the main <job-id>.log in its logging bucket — "
        f"-stdout/-stderr are empty for any pre-command failure."
    )


def shard_layout(paths, byte_sizes, n_samples, label):
    """Map each shard to the row range it covers, validated against n_samples."""
    layout, start, cum = [], 0, 0
    for q, sz in zip(paths, byte_sizes):
        assert sz % 4 == 0, f"{q}: {sz} bytes is not a whole number of float32"
        cum += sz // 4
        end = inv_tri_n(cum)
        assert tri_n(end) == cum, (
            f"{label}: {q} does not end on a row boundary ({cum} floats). "
            f"A shard may be truncated or still being written."
        )
        layout.append({"path": q, "row_start": start, "row_end": end,
                       "float_start": tri_n(start), "n_bytes": sz})
        start = end

    assert start == n_samples, (
        f"{label}: shards cover {start} rows but .grm.id has {n_samples} samples"
    )
    return pd.DataFrame(layout)


def read_grm_ids_gs(gs_path, local_path):
    if not os.path.isfile(local_path):
        r = subprocess.run(["gcloud", "storage", "cp", gs_path, local_path],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"cannot fetch {gs_path}\n{r.stderr.strip()}")
    d = pd.read_csv(local_path, sep=r"\s+", header=None, names=["FID", "IID"],
                    dtype=str)
    return d["IID"].tolist()


classic_paths, classic_bytes = require_shards(CLASSIC_GS, N_SHARDS, "classic")
classic_ids = read_grm_ids_gs(f"{CLASSIC_GS}/grm.shard1.grm.id",
                              f"{LOCAL}/classic.grm.id")
classic_layout = shard_layout(classic_paths, classic_bytes,
                              len(classic_ids), "classic")

print(f"classic: {len(classic_ids):,} samples, "
      f"{sum(classic_bytes) / 1024**3:.1f} GB across {N_SHARDS} shards "
      f"(streamed, not downloaded)")
print(classic_layout[["row_start", "row_end"]].head(3).to_string(index=False))
print("  ...")

## Download and concatenate the rung GRM

The rung triangle is ~5 GB, so it is pulled local once and concatenated into a
single file that can be memory-mapped. Ordering matters: shards must be joined in
ascending *k*, which is why they are listed explicitly rather than globbed.

In [ ]:
rung_ids_path = f"{LOCAL}/rung.grm.id"
rung_ids = read_grm_ids_gs(f"{RUNG_GS}/grm.shard1.grm.id", rung_ids_path)
n_rung = len(rung_ids)
expected_bytes = tri_n(n_rung) * 4

if os.path.isfile(RUNG_TRI) and os.path.getsize(RUNG_TRI) == expected_bytes:
    print(f"reusing {RUNG_TRI}")
else:
    require_shards(RUNG_GS, N_SHARDS, "rung")      # report before downloading
    for k in range(1, N_SHARDS + 1):
        dst = f"{LOCAL}/shard{k}.grm.bin"
        if not os.path.isfile(dst):
            subprocess.run(["gcloud", "storage", "cp",
                            f"{RUNG_GS}/grm.shard{k}.grm.bin", dst], check=True)
    rung_parts = [f"{LOCAL}/shard{k}.grm.bin" for k in range(1, N_SHARDS + 1)]
    shard_layout(rung_parts, [os.path.getsize(q) for q in rung_parts],
                 n_rung, "rung")               # validate before concatenating

    with open(RUNG_TRI, "wb") as out:
        for p in rung_parts:
            with open(p, "rb") as src:
                while chunk := src.read(64 << 20):
                    out.write(chunk)
    for p in rung_parts:
        os.remove(p)

actual = os.path.getsize(RUNG_TRI)
assert actual == expected_bytes, (
    f"{RUNG_TRI} is {actual} bytes, expected {expected_bytes} "
    f"for {n_rung} samples"
)
rung_tri = np.memmap(RUNG_TRI, dtype=np.float32, mode="r")
print(f"rung: {n_rung:,} samples, {actual / 1024**3:.2f} GB, memory-mapped")

## Align the two sample sets

The intersection is ordered by **classic** row index, so the streaming pass below
walks the classic file forward. At step *m* the pairs available are sample *m*
against samples *0…m−1*, which covers each unordered pair exactly once and never
touches the diagonal.

In [ ]:
classic_pos = {s: i for i, s in enumerate(classic_ids)}
rung_pos    = {s: i for i, s in enumerate(rung_ids)}

common = sorted(set(classic_pos) & set(rung_pos), key=classic_pos.get)
print(f"classic {len(classic_ids):,} | rung {n_rung:,} | shared {len(common):,}")

only_rung = set(rung_pos) - set(classic_pos)
if only_rung:
    print(f"NOTE: {len(only_rung):,} rung samples absent from the classic GRM "
          f"— excluded from the comparison")

if MAX_ROWS is not None and MAX_ROWS < len(common):
    rng = np.random.default_rng(RNG_SEED)
    keep = np.sort(rng.choice(len(common), size=MAX_ROWS, replace=False))
    common = [common[i] for i in keep]
    print(f"MAX_ROWS: restricted to {len(common):,} samples")

c_idx = np.array([classic_pos[s] for s in common], dtype=np.int64)
r_idx = np.array([rung_pos[s]    for s in common], dtype=np.int64)
n_common = len(common)
n_pairs_total = n_common * (n_common - 1) // 2

read_gb = (c_idx + 1).sum() * 4 / 1024**3
print(f"\n{n_pairs_total:,} pairs to compare")
print(f"~{read_gb:.1f} GB to read from the classic GRM off the bucket")

## Streaming join

One pass over the classic GRM, streamed from the bucket shard by shard. Per kept
row it accumulates, per classic bin, the count and the sum and sum-of-squares of
the rung value — enough for a mean and standard deviation. It also accumulates the
raw cross-moments, so the regression slope is exact rather than fitted to the
binned means, and keeps a random sample of pairs for the density panel.

The intersection is ordered by classic row index, which is the order the stream
delivers rows in, so at step *m* the available partners are samples *0…m−1*. That
covers every unordered pair once and never touches the diagonal.

Diagonal entries are collected separately — they are *1 + F*, on a different scale
from off-diagonal relatedness, and averaging them in would be wrong.

Expect roughly ten minutes to an hour depending on bucket throughput. The progress
line reports transferred volume so a stall is visible.

In [ ]:
def stream_classic_rows(layout, wanted):
    """
    Yield (m, row) in ascending classic row order, where wanted[m] is that row's
    index in the classic GRM. Streams via `gcloud storage cat` — nothing is
    written to disk. A pipe cannot seek, so unwanted rows are read and discarded,
    but a shard is abandoned once its last wanted row has gone past.
    """
    wanted = np.asarray(wanted)
    assert np.all(np.diff(wanted) > 0), "wanted rows must be strictly ascending"
    pos_of = {int(r): m for m, r in enumerate(wanted)}

    for sh in layout.itertuples():
        here = wanted[(wanted >= sh.row_start) & (wanted < sh.row_end)]
        if here.size == 0:
            continue
        last, need = int(here[-1]), {int(v) for v in here}

        proc = subprocess.Popen(["gcloud", "storage", "cat", sh.path],
                                stdout=subprocess.PIPE,
                                stderr=subprocess.DEVNULL)
        try:
            for i in range(sh.row_start, sh.row_end):
                nb_row = (i + 1) * 4
                buf = proc.stdout.read(nb_row)
                if len(buf) != nb_row:
                    raise IOError(
                        f"short read at row {i} of {sh.path} "
                        f"(got {len(buf)} of {nb_row} bytes) — stream interrupted"
                    )
                if i in need:
                    yield pos_of[i], np.frombuffer(buf, dtype=np.float32)
                if i == last:
                    break
        finally:
            proc.stdout.close()
            proc.terminate()
            proc.wait()


n_bin = len(BINS) - 1
b_count = np.zeros(n_bin, dtype=np.int64)
b_sum   = np.zeros(n_bin, dtype=np.float64)
b_sumsq = np.zeros(n_bin, dtype=np.float64)

m_n = 0
m_x = m_y = m_xx = m_xy = m_yy = 0.0

p_keep = min(1.0, SCATTER_TARGET / max(n_pairs_total, 1))
rng = np.random.default_rng(RNG_SEED)
sx, sy = [], []

diag_x = np.full(n_common, np.nan, dtype=np.float32)
diag_y = np.full(n_common, np.nan, dtype=np.float32)

t0, seen, bytes_read = time.time(), 0, 0
for m, row in stream_classic_rows(classic_layout, c_idx):
    seen += 1
    bytes_read += row.nbytes

    diag_x[m] = row[c_idx[m]]
    rd = r_idx[m]
    diag_y[m] = rung_tri[tri_n(rd) + rd]

    if m == 0:
        continue

    x = row[c_idx[:m]].astype(np.float64)

    hi = np.maximum(rd, r_idx[:m])
    lo = np.minimum(rd, r_idx[:m])
    y = rung_tri[hi * (hi + 1) // 2 + lo].astype(np.float64)

    ok = np.isfinite(x) & np.isfinite(y)
    if not ok.all():
        x, y = x[ok], y[ok]

    m_n  += x.size
    m_x  += x.sum();        m_y  += y.sum()
    m_xx += (x * x).sum();  m_yy += (y * y).sum()
    m_xy += (x * y).sum()

    b = np.digitize(x, BINS) - 1
    inb = (b >= 0) & (b < n_bin)
    np.add.at(b_count, b[inb], 1)
    np.add.at(b_sum,   b[inb], y[inb])
    np.add.at(b_sumsq, b[inb], y[inb] ** 2)

    if p_keep >= 1.0:
        sx.append(x); sy.append(y)
    else:
        pick = rng.random(x.size) < p_keep
        if pick.any():
            sx.append(x[pick]); sy.append(y[pick])

    if seen % 2000 == 0:
        el = time.time() - t0
        print(f"  {seen:,}/{n_common:,} rows | "
              f"{m_n/max(n_pairs_total,1)*100:5.1f}% of pairs | "
              f"{bytes_read/1024**3:5.1f} GB kept | {el/60:5.1f} min", flush=True)

assert seen == n_common, f"stream yielded {seen} rows, expected {n_common}"

scat_x = np.concatenate(sx) if sx else np.array([])
scat_y = np.concatenate(sy) if sy else np.array([])
print(f"\ndone in {(time.time()-t0)/60:.1f} min — {m_n:,} pairs compared, "
      f"{len(scat_x):,} kept for the density panel")

## Summary statistics

In [ ]:
mx, my = m_x / m_n, m_y / m_n
vx = m_xx / m_n - mx ** 2
vy = m_yy / m_n - my ** 2
cxy = m_xy / m_n - mx * my

slope     = cxy / vx
intercept = my - slope * mx
pearson   = cxy / math.sqrt(vx * vy)
slope0    = m_xy / m_xx          # through the origin

print(f"pairs compared      {m_n:,}")
print(f"mean classic        {mx: .6f}   sd {math.sqrt(vx):.6f}")
print(f"mean rung           {my: .6f}   sd {math.sqrt(vy):.6f}")
print(f"Pearson r           {pearson: .5f}")
print(f"OLS slope           {slope: .5f}  (intercept {intercept: .6f})")
print(f"slope through 0     {slope0: .5f}")
print(f"\ndiagonal: classic mean {diag_x.mean():.5f}, "
      f"rung mean {diag_y.mean():.5f}")

valid = b_count > 0
centres = (BINS[:-1] + BINS[1:]) / 2
b_mean = np.full(n_bin, np.nan)
b_sd   = np.full(n_bin, np.nan)
b_mean[valid] = b_sum[valid] / b_count[valid]
var = b_sumsq[valid] / b_count[valid] - b_mean[valid] ** 2
b_sd[valid] = np.sqrt(np.clip(var, 0, None))

binned = pd.DataFrame({"classic_bin_centre": centres, "n_pairs": b_count,
                       "rung_mean": b_mean, "rung_sd": b_sd})
print(f"\n{valid.sum()} of {n_bin} bins populated")
binned[valid].iloc[::max(1, valid.sum() // 12)].round(5)

## Plot

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], hspace=0.28, wspace=0.22)

# ---- binned means, the main panel ----
ax = fig.add_subplot(gs[0, :])
lim = [centres[valid].min(), centres[valid].max()]
ax.plot(lim, lim, color="0.6", ls="--", lw=1, label="y = x")
ax.plot(lim, [intercept + slope * v for v in lim], color="tab:red", lw=1,
        label=f"OLS slope {slope:.3f}")
ax.fill_between(centres[valid],
                (b_mean - b_sd)[valid], (b_mean + b_sd)[valid],
                color="tab:blue", alpha=0.20, lw=0, label="±1 sd within bin")
ax.scatter(centres[valid], b_mean[valid], s=np.clip(
               np.log10(b_count[valid] + 1) * 6, 2, 60),
           color="tab:blue", zorder=3, label="bin mean (size ∝ log n)")
ax.set_xlabel("classic GRM (MAF > 1 %, full cohort)")
ax.set_ylabel(f"MAF > {RUNG} GRM (binned mean)")
ax.set_title(f"Rung {RUNG} vs classic GRM — {m_n:,} pairs, "
             f"r = {pearson:.4f}", loc="left")
ax.legend(fontsize=8, loc="upper left")

# ---- pair density ----
ax2 = fig.add_subplot(gs[1, 0])
if len(scat_x):
    hb = ax2.hexbin(scat_x, scat_y, gridsize=90, bins="log", mincnt=1,
                    cmap="viridis")
    fig.colorbar(hb, ax=ax2, label="pairs (log)")
    ax2.plot(lim, lim, color="0.8", ls="--", lw=1)
ax2.set_xlabel("classic GRM"); ax2.set_ylabel(f"MAF > {RUNG} GRM")
ax2.set_title(f"raw pairs (n={len(scat_x):,})", loc="left", fontsize=10)

# ---- pairs per bin, so sparse bins are visible ----
ax3 = fig.add_subplot(gs[1, 1])
ax3.step(centres[valid], b_count[valid], where="mid", lw=1, color="0.3")
ax3.set_yscale("log")
ax3.set_xlabel("classic GRM"); ax3.set_ylabel("pairs per bin")
ax3.set_title("bin occupancy — trust the left of the top panel most",
              loc="left", fontsize=10)

plt.savefig(f"{LOCAL}/grm_compare_{RUNG}.png", dpi=150, bbox_inches="tight")
plt.show()

## Diagonal

Self-relatedness is *1 + F*, a different quantity from off-diagonal relatedness,
so it is shown on its own rather than folded into the bins above.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sub = (np.random.default_rng(RNG_SEED).choice(n_common,
           size=min(40_000, n_common), replace=False))
ax.scatter(diag_x[sub], diag_y[sub], s=2, alpha=0.15, rasterized=True,
           color="0.3")
d_lim = [min(diag_x.min(), diag_y.min()), max(diag_x.max(), diag_y.max())]
ax.plot(d_lim, d_lim, color="tab:red", ls="--", lw=1)
ax.set_xlabel("classic diagonal (1 + F)")
ax.set_ylabel(f"MAF > {RUNG} diagonal (1 + F)")
ax.set_title(f"diagonal — r = {np.corrcoef(diag_x, diag_y)[0,1]:.4f}", loc="left")
plt.tight_layout()
plt.savefig(f"{LOCAL}/grm_compare_{RUNG}_diagonal.png", dpi=150)
plt.show()

## Write results

In [ ]:
OUT_GS = f"{R_GS}/03_grm/ladder_compare"

binned_path = f"{LOCAL}/grm_compare_{RUNG}_binned.tsv"
binned[valid].to_csv(binned_path, sep="\t", index=False, float_format="%.6f")

summary_path = f"{LOCAL}/grm_compare_{RUNG}_summary.tsv"
pd.DataFrame([{
    "rung": RUNG, "n_pairs": m_n, "n_samples": n_common,
    "mean_classic": mx, "mean_rung": my,
    "sd_classic": math.sqrt(vx), "sd_rung": math.sqrt(vy),
    "pearson_r": pearson, "ols_slope": slope, "ols_intercept": intercept,
    "slope_through_origin": slope0,
    "diag_mean_classic": float(diag_x.mean()),
    "diag_mean_rung": float(diag_y.mean()),
    "diag_r": float(np.corrcoef(diag_x, diag_y)[0, 1]),
}]).to_csv(summary_path, sep="\t", index=False, float_format="%.6f")
print(open(summary_path).read())

for f in (binned_path, summary_path,
          f"{LOCAL}/grm_compare_{RUNG}.png",
          f"{LOCAL}/grm_compare_{RUNG}_diagonal.png"):
    subprocess.run(["gcloud", "storage", "cp", f, f"{OUT_GS}/"], check=True)
print(f"uploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser(
    '~/repos/AOU-covariance/notebooks/extra/grm_ladder_compare.ipynb')
subprocess.run(['gcloud', 'storage', 'cp', _nb,
                f'{OUT_GS}/notebooks/grm_ladder_compare.ipynb'], check=True)
print('notebook copied')